# RentCheck AI — Colab training notebook

Bu notebook bir dəfə **Runtime → Run all** edildikdə hər şeyi görür: repo-nu çəkir,
mühiti qurur, CarDD datasını endirib YOLO formatına çevirir, etiketləri yoxlayır,
modeli öyrədir və nəticə cədvəlini çıxarır.

**Bir dəfəlik hazırlıq (yalnız ilk dəfə):**

1. `Runtime → Change runtime type → GPU` seçin (T4 kifayətdir).
2. Sol paneldəki 🔑 **Secrets** bölməsinə iki gizli dəyər əlavə edin:
   - `KAGGLE_USERNAME`
   - `KAGGLE_KEY`

   Bunlar `kaggle.json` faylının içindəki iki dəyərdir. Hər ikisinin yanındakı
   "Notebook access" açarını aktiv edin.
3. Başqa heç nə lazım deyil.

**Sessiya kəsilsə:** narahat olmayın. Checkpoint-lər Google Drive-da saxlanılır,
notebook-u yenidən **Run all** etdikdə training yarımçıq qaldığı epoch-dan davam
edir, sıfırdan başlamır.

## 1. GPU yoxlaması

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("GPU aktiv deyil. Runtime -> Change runtime type -> GPU secin.")

## 2. Google Drive

Drive iki şey üçün lazımdır: **checkpoint-lər** (sessiya kəsilsə itməsin) və
**yekun nəticələr**. Training datası Drive-dan oxunmur — o, Colab-ın öz lokal
diskinə kopyalanır, çünki Drive minlərlə kiçik faylın oxunmasında çox yavaşdır.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/rentcheck-ai')
(DRIVE / 'runs').mkdir(parents=True, exist_ok=True)
(DRIVE / 'results').mkdir(parents=True, exist_ok=True)
print("Drive hazir:", DRIVE)

## 3. Repo

In [ ]:
import subprocess

REPO = Path('/content/rentcheck-ai')
REPO_URL = 'https://github.com/nigarrustamova/rentcheck-ai.git'
BRANCH = 'training-pipeline'   # main-e birlesdirdikden sonra burani 'main' edin

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=False)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=False)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)

print(subprocess.run(['git', '-C', str(REPO), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)

## 4. Kitabxanalar

Colab-da torch onsuz da quraşdırılıb, ona görə yalnız `ultralytics` və bir neçə
kiçik paket əlavə olunur — bu, hər sessiyada 3-4 dəqiqə qazandırır.

Tam pinlənmiş mühit lazım olsa (məsələn yekun rəqəmləri Colab-da alsanız),
aşağıdakı şərh sətrini açın. Onda torch da yenidən quraşdırılacaq və
`requirements.txt`-dəki dəqiq versiyalar işlədiləcək.

In [ ]:
!pip install -q ultralytics==8.4.128 pycocotools

# Tam pinlenmis muhit ucun asagidaki setri acin:
# !pip install -q -r /content/rentcheck-ai/requirements.txt

import torch, ultralytics
print("torch:", torch.__version__)
print("ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 5. CarDD datası

Data Kaggle-dan endirilir və Colab-ın lokal diskinə (`/content`) açılır.

**Lisenziya qeydi:** CarDD-nin öz lisenziyası dataseti üçüncü şəxslərə paylamağı
qadağan edir. Bu Kaggle nüsxəsi prototip işi üçündür. Müəlliflərdən rəsmi icazə
gəldikdən sonra yekun nəticələr rəsmi nüsxə ilə alınmalı, paperdə mənbə kimi rəsmi
kanal göstərilməlidir.

In [ ]:
import os, shutil

RAW = Path('/content/data/raw')
CARDD = RAW / 'CarDD_release' / 'CarDD_COCO'

if CARDD.exists():
    print("Data artiq yerindedir:", CARDD)
else:
    from google.colab import userdata
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except Exception as exc:
        raise SystemExit(
            "Kaggle acarlari tapilmadi. Sol paneldeki Secrets bolmesine "
            "KAGGLE_USERNAME ve KAGGLE_KEY elave edin, her ikisi ucun "
            "'Notebook access' aktiv olsun."
        ) from exc

    RAW.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d nasimetemadi/car-damage-detection -p {RAW} --unzip -q
    print("endirildi")

assert CARDD.exists(), f"Gozlenilen qovluq tapilmadi: {CARDD}"
!ls {CARDD}

## 6. COCO → YOLO çevirmə və yoxlama

In [ ]:
%cd /content/rentcheck-ai

!python src/data/coco_to_yolo.py --src {CARDD} --dst /content/data/processed
!python src/data/check_dataset.py --root {CARDD}

Aşağıdakı xana çevrilmiş etiketləri şəkillərin üstünə çəkib göstərir. Koordinat
səhvləri səssiz olur — model yanlış yerdəki maskalardan da "öyrənir" və problem
yalnız ən sonda üzə çıxır. Bir dəfə gözlə baxmaq bunun qarşısını alır.

In [ ]:
from IPython.display import Image as ShowImage, display

!python src/data/visualize_labels.py --data /content/data/processed/cardd_seg --split val -n 3

for path in sorted(Path('report/figures/label_check').glob('*.jpg'))[:3]:
    display(ShowImage(filename=str(path), width=600))

## 7. Training

Checkpoint-lər Drive-a yazılır. Əvvəlki sessiya yarımçıq qalıbsa, bu xana onu
avtomatik tapıb **davam etdirir**.

Batch ölçüsü GPU yaddaşına görə seçilir: 16 GB-dan az olsa 8, yoxsa 16.

In [ ]:
runs_local = REPO / 'runs'
if runs_local.exists() and not runs_local.is_symlink():
    shutil.rmtree(runs_local)
if not runs_local.exists():
    runs_local.symlink_to(DRIVE / 'runs')
print("runs ->", runs_local.resolve())

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
batch = 16 if gpu_gb >= 15 else 8
print(f"GPU {gpu_gb:.0f} GB -> batch {batch}")

last = DRIVE / 'runs' / 'main_seg' / 'weights' / 'last.pt'
resume = '--resume' if last.exists() else ''
if resume:
    print("Yarimciq qacis tapildi, davam edilir.")

!python src/train/train.py --config configs/main.yaml --device 0 --batch {batch} {resume}

## 8. Qiymətləndirmə

Validation bölgüsü üzərində per-class cədvəl. **Test bölgüsünə burada
toxunulmur** — o, yalnız yekun rəqəmlər üçündür və bir dəfə istifadə olunur.

In [ ]:
!python src/eval/evaluate.py --weights runs/main_seg/weights/best.pt --split val --device 0

import pandas as pd
csv_path = Path('report/results/main_seg_val.csv')
if csv_path.exists():
    display(pd.read_csv(csv_path).round(3))

## 9. Nəticələri Drive-a köçürmək

In [ ]:
results_dir = Path('report/results')
if results_dir.exists():
    for src in results_dir.glob('*'):
        shutil.copy2(src, DRIVE / 'results' / src.name)

print("Drive-a kopyalandi:")
for f in sorted((DRIVE / 'results').iterdir()):
    print("  ", f.name)
print("\nCekiler:", DRIVE / 'runs' / 'main_seg' / 'weights')

## Əlavə qaçışlar

Baseline və ablation eyni qaydada işlədilir, sadəcə konfiq adı dəyişir:

```
!python src/train/train.py --config configs/baseline.yaml --device 0 --batch 16
!python src/train/train.py --config configs/ablation_noaug.yaml --device 0 --batch 16
```

Hər biri öz `runs/<ad>/` qovluğuna yazır, bir-birini əvəz etmir.

## Sessiya kəsilsə

Notebook-u yenidən açıb **Run all** edin. Data yenidən endiriləcək (bir neçə
dəqiqə), amma training yarımçıq qaldığı yerdən davam edəcək — 7-ci xana bunu
özü həll edir.